## 0 — Imports

In [1]:
import os, json, math, random, time, tqdm
from pathlib import Path
from typing import Dict, Tuple, List, Optional
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm, cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score, confusion_matrix)
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("clf512")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a")

/home/dll0706/Documents/Bakwowi_Junior_CV_Project/.conda_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda | GPU: NVIDIA RTX A5000


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve,
)

import matplotlib
matplotlib.use('Agg')   # safe for notebook + server both
import matplotlib.pyplot as plt
import seaborn as sns

## 1 — CONFIG  (paths to your workstation; input_size = 512)

In [4]:
CFG = {
    # ── Paths ──────────────────────────────────────────────────────────────
    "annotation_csv": "./1st-krones-vision-ai-challenge/train.csv",
    "image_dir":      "./1st-krones-vision-ai-challenge/train_images",
    "coco_json":      "./1st-krones-vision-ai-challenge/train_annotations.json",
    "test_dir":       "./1st-krones-vision-ai-challenge/test_images",
    "test_coco_json": "./1st-krones-vision-ai-challenge/test_annotations_roi_only.json",
    "output_dir":     "./1st-krones-vision-ai-challenge/outputs_clf512_v2",
    "predictions_csv": "./1st-krones-vision-ai-challenge/predictions.csv",  # from a PREVIOUS run (optional)

    # ── Columns ────────────────────────────────────────────────────────────
    "label_col": "target", "image_col": "image_id",

    # ── Image / ROI ────────────────────────────────────────────────────────
    "input_size": 512,                 # ← the only deliberate change vs the 0.96 run
    "roi_category_id": 22,
    "fixed_cx": 640, "fixed_cy": 512, "fixed_radius": 450,   # fallback if image not in ROI map
    "roi_margin": 20, "clahe_clip": 2.0, "clahe_grid": (8, 8),
    "img_mean": [0.485,0.456,0.406], "img_std": [0.229,0.224,0.225],  # overwritten by timm cfg

    # ── Splits ─────────────────────────────────────────────────────────────
    "train_frac": 0.70, "val_frac": 0.15,

    # ── Model (same as 0.96 run) ───────────────────────────────────────────
    "backbone": "convnext_small", "use_msff": True, "use_cbam": True, "gem_p": 3.0,
    "hidden_dim": 512, "head_hidden": 256, "dropout1": 0.5, "dropout2": 0.2,

    # ── Loss (same) ─────────────────────────────────────────────────────────
    "focal_gamma": 3.0, "focal_alpha": 0.35,
    "fp_weight": 4.0, "fn_weight": 1.5, "hard_example_weight": 3.0,

    # ── Training (same schedule; batch reduced for 512px) ───────────────────
    "warmup_epochs": 4, "batch_size": 24, "accum_steps": 3,   # eff. batch ~72 (was 64 @320)
    "phase1_epochs": 12, "phase2_epochs": 18, "phase3_epochs": 10, "phase4_epochs": 10,
    "total_epochs": 50,
    "lr_head": 1e-3, "lr_phase2": 8e-5, "lr_phase3_bb": 5e-6, "lr_phase3_hd": 3e-5,
    "lr_phase4": 1e-6, "min_lr": 1e-7, "weight_decay": 5e-4, "grad_clip": 1.0,
    "amp": True, "num_workers": 4,

    # ── Early stop / threshold ───────────────────────────────────────────────
    "early_stop_patience": 20, "early_stop_min_delta": 5e-4,
    "min_recall_faulty": 0.99, "default_threshold": 0.5,
}
os.makedirs(CFG["output_dir"], exist_ok=True)
CFG["checkpoint_path"] = os.path.join(CFG["output_dir"], "best_clf512.pt")  # a FILE, not a dir
assert not os.path.isdir(CFG["checkpoint_path"]), "checkpoint_path is a directory — remove it"
print("CFG ready | input_size =", CFG["input_size"], "| ckpt =", CFG["checkpoint_path"])

CFG ready | input_size = 512 | ckpt = ./1st-krones-vision-ai-challenge/outputs_clf512_v2/best_clf512.pt


In [3]:
# import shutil
# # shutil.rmtree('/kaggle/working/outputs_clf512')
# shutil.copy('/kaggle/input/models/bakwowijunior/cnn-model/pytorch/default/1/best_clf512.pt', '/kaggle/working/outputs_clf512')

'/kaggle/working/outputs_clf512/best_clf512.pt'

## 2 — COCO ROI map (category 22 → cx,cy,radius), used to crop

In [5]:
def _circle_from_bbox(bbox):
    x,y,w,h = bbox
    return int(x+w/2), int(y+h/2), int(max(w,h)/2)

def _circle_from_segmentation(seg):
    pts = np.array(seg[0], dtype=np.float32).reshape(-1,2)
    cx,cy = pts[:,0].mean(), pts[:,1].mean()
    r = np.sqrt(((pts-[cx,cy])**2).sum(1)).max()
    return int(cx), int(cy), int(r)

def load_coco_roi_map(annotation_path, roi_category_id=22):
    with open(annotation_path) as f: coco = json.load(f)
    id_to_file = {im["id"]: Path(im["file_name"]).name for im in coco["images"]}
    roi = {}; skipped = 0
    for ann in coco["annotations"]:
        if ann.get("category_id") != roi_category_id: continue
        fn = id_to_file.get(ann["image_id"])
        if fn is None: continue
        if ann.get("segmentation"):
            try: roi[fn] = _circle_from_segmentation(ann["segmentation"]); continue
            except Exception: pass
        if ann.get("bbox") and len(ann["bbox"])==4:
            roi[fn] = _circle_from_bbox(ann["bbox"])
        else: skipped += 1
    log.info("ROI map: %d images | skipped %d", len(roi), skipped)
    return roi

roi_map = load_coco_roi_map(CFG["coco_json"], CFG["roi_category_id"]) if Path(CFG["coco_json"]).exists() else {}
print("ROI entries:", len(roi_map))

18:46:21 | ROI map: 35342 images | skipped 0


ROI entries: 35342


## 3 — ROI crop + CLAHE (applied inside the dataset before transforms)

In [6]:
def roi_crop_clahe(path, cfg, roi_map):
    bgr = cv2.imread(str(path))
    if bgr is None: raise FileNotFoundError(path)
    fn = Path(path).name
    cx,cy,r = roi_map.get(fn, (cfg["fixed_cx"], cfg["fixed_cy"], cfg["fixed_radius"]))
    m = cfg["roi_margin"]; h,w = bgr.shape[:2]
    x1,y1 = max(0,cx-r-m), max(0,cy-r-m); x2,y2 = min(w,cx+r+m), min(h,cy+r+m)
    crop = bgr[y1:y2, x1:x2]
    if crop.size == 0: crop = bgr
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB)
    cl = cv2.createCLAHE(clipLimit=cfg["clahe_clip"], tileGridSize=tuple(cfg["clahe_grid"]))
    lab[:,:,0] = cl.apply(lab[:,:,0])
    rgb = cv2.cvtColor(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
    H,W = rgb.shape[:2]; s = max(H,W)            # square-pad so resize doesn't distort
    pad = np.zeros((s,s,3), dtype=rgb.dtype)
    pad[(s-H)//2:(s-H)//2+H, (s-W)//2:(s-W)//2+W] = rgb
    return pad

## 4 — Augmentations (identical recipe to the 0.96 run)

In [7]:
# def build_train_transforms(cfg):
#     s = cfg["input_size"]
#     return A.Compose([
#         A.Resize(s,s), A.Rotate(limit=180,p=1.0),
#         A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
#         A.RandomResizedCrop(size=(s,s), scale=(0.88,1.0), ratio=(0.97,1.03), p=0.4),
#         A.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05, hue=0.01, p=0.6),
#         A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.4),
#         A.Sharpen(alpha=(0.1,0.3), lightness=(0.9,1.1), p=0.3),
#         A.GaussianBlur(blur_limit=3, p=0.25),
#         A.GaussNoise(p=0.2),
#         A.GridDistortion(num_steps=5, distort_limit=0.10, p=0.15),
#         A.Normalize(mean=cfg["img_mean"], std=cfg["img_std"]), ToTensorV2(),
#     ])
# def build_eval_transforms(cfg):
#     s = cfg["input_size"]
#     return A.Compose([A.Resize(s,s), A.Normalize(mean=cfg["img_mean"], std=cfg["img_std"]), ToTensorV2()])

def build_train_transforms(cfg):
    s = cfg["input_size"]
    return A.Compose([
        # Bilateral + CLAHE done upstream in roi_crop_clahe (not repeated here).
        A.Resize(s, s),
        # # Geometric — matches real test-time variation, safe for symmetric bases
        A.Rotate(limit=180, p=1.0),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.RandomResizedCrop(size=(s, s), scale=(0.92, 1.0), ratio=(0.98, 1.02), p=0.3),
        # Photometric — keep SMALL; line lighting varies a little, not a lot
        A.RandomBrightnessContrast(brightness_limit=0.06, contrast_limit=0.06, p=0.3),
        # CUT: ColorJitter (no real color signal — invents OOD color)
        # CUT: Sharpen (interacts badly with CLAHE, can fake defect edges)
        # CUT/min: GaussNoise — the worst offender for faint specks
        # A.GaussNoise(var_limit=(2.0, 8.0), p=0.10),   # if kept at all: tiny and rare
        A.GaussNoise(std_range=(0.004, 0.02), p=0.50),   # even smaller and rarer
        # CUT: GaussianBlur (blurs away the 13px specks you need)
        # CUT: GridDistortion (warps small defects unpredictably)
        A.Normalize(mean=cfg["img_mean"], std=cfg["img_std"]), ToTensorV2(),
    ])

def build_eval_transforms(cfg):
    s = cfg["input_size"]
    return A.Compose([A.Resize(s,s), A.Normalize(mean=cfg["img_mean"], std=cfg["img_std"]), ToTensorV2()])

In [71]:
print(A.__version__)

2.0.8


In [7]:
import numpy as np, torch

def denorm_for_display(t, cfg):
    """Invert A.Normalize so a post-transform tensor displays in true color (gray here)."""
    mean = np.array(cfg["img_mean"]).reshape(3, 1, 1)
    std  = np.array(cfg["img_std"]).reshape(3, 1, 1)
    img = t.detach().cpu().numpy() * std + mean      # undo (x-mean)/std
    img = np.clip(img.transpose(1, 2, 0), 0, 1)      # CHW->HWC, clip to valid range
    return img

In [10]:
# view some samples using opencv and matplotlib to verify ROI cropping and augmentations
import matplotlib.pyplot as plt
# import cv2

df = pd.read_csv(CFG["annotation_csv"])
sample_df = df.sample(6, random_state=SEED).reset_index(drop=True)
train_transforms = build_train_transforms(CFG)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (_, row) in enumerate(sample_df.iterrows()):
    img_path = os.path.join(CFG["image_dir"], row[CFG["image_col"]])
    img = roi_crop_clahe(img_path, CFG, roi_map)
    augmented = denorm_for_display(train_transforms(image=img)["image"], CFG)
    axes[i//3, i%3].imshow(augmented)
    axes[i//3, i%3].set_title(f"Label: {row[CFG['label_col']]}")
    axes[i//3, i%3].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "sample_augmentations.png"))
plt.show()


In [39]:
import cv2, matplotlib.pyplot as plt
p = "1st-krones-vision-ai-challenge/train_images/0a0b08c7-183b-4905-aec7-44371a16b8e4_000000008768.png"
raw = cv2.imread(p)                          # BGR
gray_check = raw[:,:,0].astype(int) - raw[:,:,2].astype(int)
print("B-R channel diff (should be ~0 for grayscale):", gray_check.mean())

fig, ax = plt.subplots(1,3, figsize=(12,4))
ax[0].imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)); ax[0].set_title("raw, BGR->RGB (correct)")
ax[1].imshow(raw); ax[1].set_title("raw, no convert (will look blue if bug)")
out = roi_crop_clahe(p, CFG, roi_map if 'roi_map' in dir() else {})
ax[2].imshow(out); ax[2].set_title("roi_crop_clahe output")
plt.show()

B-R channel diff (should be ~0 for grayscale): 0.0


## 5 — Dataset (ROI-cropped) + per-sample weights + dataloaders

In [8]:
class BottleDataset(Dataset):
    def __init__(self, df, transform, cfg, roi_map):
        self.df = df.reset_index(drop=True); self.transform = transform
        self.cfg = cfg; self.roi_map = roi_map
        self.has_w = "sample_weight" in self.df.columns
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        col = self.cfg["image_col"]
        path = row[col] if os.path.isabs(str(row[col])) else os.path.join(self.cfg["image_dir"], row[col])
        img = roi_crop_clahe(path, self.cfg, self.roi_map)          # ← ROI crop here
        x = self.transform(image=img)["image"]
        y = torch.tensor(float(row["binary_label"]), dtype=torch.float32)
        w = torch.tensor(float(row["sample_weight"]) if self.has_w else 1.0, dtype=torch.float32)
        return x, y, w

def assign_sample_weights(df, cfg):
    """FP/FN weighting from a previous predictions.csv if present; else uniform.
    Mirrors the 0.96 run: GOOD predicted FAULTY -> fp_weight; FAULTY predicted GOOD -> fn_weight."""
    df = df.copy(); df["sample_weight"] = 1.0
    p = cfg.get("predictions_csv")
    if p and Path(p).exists():
        pred = pd.read_csv(p)
        key = cfg["image_col"] if cfg["image_col"] in pred.columns else pred.columns[0]
        pred_map = dict(zip(pred[key], pred.get("pred_label", pred.iloc[:,-1])))
        def w(row):
            pl = pred_map.get(row[cfg["image_col"]])
            if pl is None: return 1.0
            if row["binary_label"]==0 and pl==1: return cfg["fp_weight"]
            if row["binary_label"]==1 and pl==0: return cfg["fn_weight"]
            return 1.0
        df["sample_weight"] = df.apply(w, axis=1)
        log.info("Sample weights from predictions.csv | mean=%.3f", df["sample_weight"].mean())
    else:
        log.info("No predictions.csv — uniform weights (first run).")
    return df

def make_loaders(df, cfg, roi_map):
    vt = 1.0 - cfg["train_frac"]; rel = cfg["val_frac"]/vt
    tr, tmp = train_test_split(df, test_size=vt, stratify=df["binary_label"], random_state=SEED)
    va, te = train_test_split(tmp, test_size=1-rel, stratify=tmp["binary_label"], random_state=SEED)
    log.info("Split — train %d val %d test %d", len(tr), len(va), len(te))
    tr = assign_sample_weights(tr, cfg)              # weights on TRAIN only
    counts = np.bincount(tr["binary_label"].values)
    sw = (1.0/counts)[tr["binary_label"].values]
    sampler = WeightedRandomSampler(sw, num_samples=len(sw), replacement=True)
    bs, nw = cfg["batch_size"], cfg["num_workers"]
    trl = DataLoader(BottleDataset(tr, build_train_transforms(cfg), cfg, roi_map),
                     batch_size=bs, sampler=sampler, num_workers=nw, pin_memory=True,
                     drop_last=True, persistent_workers=nw>0)
    val = DataLoader(BottleDataset(va, build_eval_transforms(cfg), cfg, roi_map),
                     batch_size=bs*2, shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=nw>0)
    tel = DataLoader(BottleDataset(te, build_eval_transforms(cfg), cfg, roi_map),
                     batch_size=bs*2, shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=nw>0)
    return trl, val, tel

df = pd.read_csv(CFG["annotation_csv"])
df["binary_label"] = df[CFG["label_col"]].astype(int)
if CFG["image_col"] not in df.columns:
    for alt in tqdm.tqdm(["image_id","file_name","filename","image"]):
        if alt in df.columns: df[CFG["image_col"]] = df[alt]; break
print("Rows:", len(df), "| FAULTY:", int(df['binary_label'].sum()), f"({100*df['binary_label'].mean():.1f}%)")
train_loader, val_loader, test_loader = make_loaders(df, CFG, roi_map)

# sampler balance check
g=f=0
for _,y,_ in train_loader:
    g+=(y==0).sum().item(); f+=(y==1).sum().item()
    if g+f>=720: break
print(f"Sampler check — GOOD {100*g/(g+f):.0f}% / FAULTY {100*f/(g+f):.0f}%")

18:46:56 | Split — train 24739 val 5301 test 5302


18:46:56 | Sample weights from predictions.csv | mean=1.000


Rows: 35342 | FAULTY: 20613 (58.3%)
Sampler check — GOOD 52% / FAULTY 48%


## 6 — Architecture: GeM (fp16-safe) + CBAM + MSFF (identical to 0.96 run)

In [9]:
import timm, torch
_bb = timm.create_model(CFG["backbone"], pretrained=False,
                        num_classes=0, global_pool="",
                        features_only=True, out_indices=(0,1,2,3))
_x = torch.randn(1, 3, CFG["input_size"], CFG["input_size"])
with torch.no_grad():
    feats = _bb(_x)
for i, f in enumerate(feats):
    print(f"stage {i}: channels={f.shape[1]:4d}  spatial={f.shape[2]}x{f.shape[3]}")
STAGE_DIMS = [f.shape[1] for f in feats]
print("STAGE_DIMS =", STAGE_DIMS)
del _bb, feats

stage 0: channels=  96  spatial=128x128
stage 1: channels= 192  spatial=64x64
stage 2: channels= 384  spatial=32x32
stage 3: channels= 768  spatial=16x16
STAGE_DIMS = [96, 192, 384, 768]


In [10]:
class GeMPooling(nn.Module):
    """fp16-safe GeM; p clamped [1,5] and pow done in fp32 to prevent NaN under AMP."""
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.ones(1)*p); self.eps = eps
    def forward(self, x):
        p = self.p.clamp(1.0, 5.0)
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            xf = x.float().clamp(min=self.eps)
            out = F.avg_pool2d(xf.pow(p), kernel_size=(xf.shape[-2], xf.shape[-1])).pow(1.0/p)
        return out.view(out.shape[0], -1)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__(); mid=max(channels//reduction,8)
        self.fc = nn.Sequential(nn.Linear(channels,mid,bias=False), nn.ReLU(inplace=True),
                                nn.Linear(mid,channels,bias=False))
    def forward(self,x):
        avg=x.mean(dim=[2,3]); mx=x.flatten(2).max(dim=2).values
        return x*torch.sigmoid(self.fc(avg)+self.fc(mx)).unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__(); self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False)
    def forward(self,x):
        a=x.mean(dim=1,keepdim=True); m=x.max(dim=1,keepdim=True).values
        return x*torch.sigmoid(self.conv(torch.cat([a,m],dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_k=7):
        super().__init__(); self.c=ChannelAttention(channels,reduction); self.s=SpatialAttention(spatial_k)
    def forward(self,x): return self.s(self.c(x))

class MultiScaleFeatureFusion(nn.Module):
    def __init__(self, stage_dims=[192, 384, 768], out_dim=512, gem_p=3.0):
        super().__init__()
        # Instead of instantly flat-pooling to 1x1, align all layers to a unified intermediate spatial grid 
        # This keeps spatial localization data intact for fine scuffs and micro-contamination.
        self.target_grid = (32, 32) 
        
        # 1x1 convolutions to compress channel weights safely before cross-layer blending
        self.channel_projs = nn.ModuleList([
            nn.Conv2d(dim, out_dim // len(stage_dims), kernel_size=1) for dim in stage_dims
        ])
        
        # Final global pooling layer over the blended spatial map
        self.gem = GeMPooling(gem_p)
        self.final_proj = nn.Sequential(
            nn.Linear((out_dim // len(stage_dims)) * len(stage_dims), out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU()
        )

    def forward(self, feats):
        aligned_feats = []
        for i, f in enumerate(feats):
            # 1. Project channel dimensions down
            proj_f = self.channel_projs[i](f)
            
            # 2. Synchronize spatial footprints to standard (16x16) size smoothly
            if proj_f.shape[-2:] != self.target_grid:
                proj_f = F.adaptive_avg_pool2d(proj_f, self.target_grid)
                
            aligned_feats.append(proj_f)
            
        # 3. Concatenate layers while preserving spatial alignment maps! 
        fused_spatial = torch.cat(aligned_feats, dim=1) # Shape: [Batch, Channels, 16, 16]
        
        # 4. Collapse to a 1D vector ONLY after high-freq details and high-level semantics have blended spatially
        pooled = self.gem(fused_spatial)
        return self.final_proj(pooled)


def build_model(cfg):
    name = cfg["backbone"]
    dc = timm.data.resolve_model_data_config(timm.create_model(name, pretrained=False))
    cfg["img_mean"], cfg["img_std"] = list(dc["mean"]), list(dc["std"])
    gem_p, embed = cfg["gem_p"], cfg["hidden_dim"]
    if cfg["use_msff"]:
        # CHANGED: add stage 0 (finest map). Read dims from the backbone, don't hardcode.
        out_indices = (0, 1, 2, 3)
        backbone = timm.create_model(name, pretrained=True, num_classes=0, global_pool="",
                                     features_only=True, out_indices=out_indices)
        stage_dims = list(backbone.feature_info.channels())   # e.g. [96,192,384,768]
        cbams = nn.ModuleList([CBAM(d) if (cfg["use_cbam"] and i > 0) else nn.Identity() for i, d in enumerate(stage_dims)])
        msff = MultiScaleFeatureFusion(stage_dims, embed, gem_p)
        head = nn.Sequential(nn.Dropout(cfg["dropout1"]), nn.Linear(embed,cfg["head_hidden"]),
                             nn.BatchNorm1d(cfg["head_hidden"]), nn.ReLU(inplace=True),
                             nn.Dropout(cfg["dropout2"]), nn.Linear(cfg["head_hidden"],1))
        class Net(nn.Module):
            def __init__(s): super().__init__(); s.backbone=backbone; s.cbams=cbams; s.msff=msff; s.head=head
            def forward(s,x):
                st=s.backbone(x); at=[c(f) for c,f in zip(s.cbams,st)]; return s.head(s.msff(at))
        model = Net()
    else:
        backbone = timm.create_model(name, pretrained=True, num_classes=0, global_pool="")
        cbam = CBAM(768) if cfg["use_cbam"] else nn.Identity(); gem=GeMPooling(gem_p)
        head = nn.Sequential(nn.Dropout(cfg["dropout1"]), nn.Linear(768,embed), nn.BatchNorm1d(embed),
                             nn.ReLU(inplace=True), nn.Dropout(cfg["dropout2"]), nn.Linear(embed,1))
        class Net(nn.Module):
            def __init__(s): super().__init__(); s.backbone=backbone; s.cbam=cbam; s.gem=gem; s.head=head
            def forward(s,x): return s.head(s.gem(s.cbam(s.backbone.forward_features(x))))
        model = Net()
    for m in model.modules():
        if isinstance(m,nn.Linear) and m.out_features==1:
            nn.init.kaiming_uniform_(m.weight, nonlinearity="relu"); nn.init.zeros_(m.bias)
    model = model.to(DEVICE)
    log.info("Model %s CBAM=%s MSFF=%s GeM=%.1f | params %s", name, cfg["use_cbam"], cfg["use_msff"],
             gem_p, f"{sum(p.numel() for p in model.parameters()):,}")
    return model

model = build_model(CFG)
_x = torch.randn(2,3,CFG["input_size"],CFG["input_size"]).to(DEVICE)
with torch.no_grad(): print("forward OK, out:", tuple(model(_x).shape))

18:47:17 | Loading pretrained weights from Hugging Face hub (timm/convnext_small.in12k_ft_in1k)
18:47:20 | HTTP Request: HEAD https://huggingface.co/timm/convnext_small.in12k_ft_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
18:47:20 | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
18:47:20 | [timm/convnext_small.in12k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
18:47:21 | Model convnext_small CBAM=True MSFF=True GeM=3.0 | params 50,130,824


forward OK, out: (2, 1)


In [11]:
model = build_model(CFG)
_x = torch.randn(2, 3, CFG["input_size"], CFG["input_size"]).to(DEVICE)
with torch.no_grad():
    feats = model.backbone(_x)
print("stage dims:", [f.shape[1] for f in feats], "spatial:", [tuple(f.shape[2:]) for f in feats])
print("n cbams:", len(model.cbams), "| n channel_projs:", len(model.msff.channel_projs))
out = model(_x)
print("output:", tuple(out.shape))    # MUST be (2,1)

18:47:25 | Loading pretrained weights from Hugging Face hub (timm/convnext_small.in12k_ft_in1k)
18:47:26 | HTTP Request: HEAD https://huggingface.co/timm/convnext_small.in12k_ft_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
18:47:26 | [timm/convnext_small.in12k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
18:47:26 | Model convnext_small CBAM=True MSFF=True GeM=3.0 | params 50,130,824


stage dims: [96, 192, 384, 768] spatial: [(128, 128), (64, 64), (32, 32), (16, 16)]
n cbams: 4 | n channel_projs: 4
output: (2, 1)


In [12]:
import torch.nn.functional as F

# Pull the real backbone feature shapes
backbone = model.backbone
_x = torch.randn(2, 3, CFG["input_size"], CFG["input_size"]).to(DEVICE)
with torch.no_grad():
    feats = backbone(_x)
print("=== backbone stage outputs ===")
for i, f in enumerate(feats):
    print(f"  stage {i}: channels={f.shape[1]:4d}  spatial={f.shape[2]}x{f.shape[3]}")

# Now trace through the MSFF
msff = model.msff
print("\n=== MSFF internals ===")
print("target_grid:", msff.target_grid)
print("n channel_projs:", len(msff.channel_projs))
for i, proj in enumerate(msff.channel_projs):
    print(f"  proj {i}: in={proj.in_channels} out={proj.out_channels}")

with torch.no_grad():
    # replicate forward, checking each step
    cbam_out = [c(f) for c, f in zip(model.cbams, feats)]
    aligned = []
    for i, f in enumerate(cbam_out):
        pf = msff.channel_projs[i](f)
        if pf.shape[-2:] != msff.target_grid:
            pf = F.adaptive_avg_pool2d(pf, msff.target_grid)
        aligned.append(pf)
        print(f"  aligned {i}: {tuple(pf.shape)}")
    fused = torch.cat(aligned, dim=1)
    print("  fused (concat):", tuple(fused.shape), "  <- channels should = sum of proj outs")
    pooled = msff.gem(fused)
    print("  after GeM:", tuple(pooled.shape))
    final = msff.final_proj(pooled)
    print("  after final_proj:", tuple(final.shape), "  <- MUST match head input (embed =", CFG["hidden_dim"], ")")

=== backbone stage outputs ===
  stage 0: channels=  96  spatial=128x128
  stage 1: channels= 192  spatial=64x64
  stage 2: channels= 384  spatial=32x32
  stage 3: channels= 768  spatial=16x16

=== MSFF internals ===
target_grid: (32, 32)
n channel_projs: 4
  proj 0: in=96 out=128
  proj 1: in=192 out=128
  proj 2: in=384 out=128
  proj 3: in=768 out=128
  aligned 0: (2, 128, 32, 32)
  aligned 1: (2, 128, 32, 32)
  aligned 2: (2, 128, 32, 32)
  aligned 3: (2, 128, 32, 32)
  fused (concat): (2, 512, 32, 32)   <- channels should = sum of proj outs
  after GeM: (2, 512)
  after final_proj: (2, 512)   <- MUST match head input (embed = 512 )


In [ ]:
actual_dims = [f.shape[1] for f in feats]
print("hardcoded stage_dims:", [192, 384, 768])
print("actual backbone dims:", actual_dims)
assert actual_dims == [192, 384, 768], (
    f"MISMATCH: backbone returns {actual_dims}, but CBAM/MSFF were built for [192,384,768]. "
    f"Fix stage_dims in build_model to {actual_dims}.")
print("OK — stage dims match.")

In [13]:
actual_dims = [f.shape[1] for f in feats]
print("actual backbone dims:", actual_dims)

# After adding stage 0, expect 4 stages. Confirm the model was built to match.
expected = list(model.backbone.feature_info.channels())
print("model feature_info dims:", expected)
assert actual_dims == expected, f"MISMATCH: forward gives {actual_dims}, feature_info says {expected}"
assert len(actual_dims) == 4, f"Expected 4 stages after adding stage 0, got {len(actual_dims)}"
assert len(model.cbams) == len(actual_dims), f"{len(model.cbams)} CBAMs for {len(actual_dims)} stages"
assert len(model.msff.channel_projs) == len(actual_dims), "MSFF proj count != stage count"
print(f"OK — {len(actual_dims)} stages, CBAM/MSFF aligned.")

actual backbone dims: [96, 192, 384, 768]
model feature_info dims: [96, 192, 384, 768]
OK — 4 stages, CBAM/MSFF aligned.


In [14]:
print("n cbams:", len(model.cbams), "| n channel_projs:", len(model.msff.channel_projs))
out = model(_x)
print("output:", tuple(out.shape))   # must be (2,1) with no dim error

n cbams: 4 | n channel_projs: 4
output: (2, 1)


## 7 — Freeze helpers (4-phase progressive unfreeze, same as 0.96 run)

In [15]:
def freeze_backbone(model):
    for p in model.backbone.parameters(): p.requires_grad=False
    log.info("Backbone frozen (head only).")

def unfreeze_last_n(model, n=None):
    bb = model.backbone
    if n is None:
        for p in bb.parameters(): p.requires_grad=True
        log.info("Full backbone unfrozen.")
    elif hasattr(bb,"stages") or hasattr(bb,"stages_"):
        stages = list((bb.stages if hasattr(bb,"stages") else bb.stages_).children())
        tot=len(stages)
        for i,s in enumerate(stages):
            for p in s.parameters(): p.requires_grad = (i>=tot-n)
    else:
        # features_only wrapper: unfreeze everything as fallback
        for p in bb.parameters(): p.requires_grad=True
    for p in model.head.parameters(): p.requires_grad=True
    if hasattr(model,"msff"):
        for p in model.msff.parameters(): p.requires_grad=True
    if hasattr(model,"cbams"):
        for p in model.cbams.parameters(): p.requires_grad=True
    log.info("Trainable params: %s", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 8 — Focal loss (weighted) + NaN-guarded epoch + evaluate

In [16]:
def focal_loss(logits, targets, gamma=3.0, alpha=0.35, sample_weights=None):
    logits=logits.view(-1); targets=targets.view(-1)
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits); pt = p*targets + (1-p)*(1-targets)
    at = alpha*targets + (1-alpha)*(1-targets)
    loss = at*(1-pt)**gamma*bce
    if sample_weights is not None: loss = loss*sample_weights.view(-1)
    return loss.mean()

def train_one_epoch(model, loader, optimizer, scaler, cfg, apply_weights=True):
    model.train(); running=0.0; n=0; skipped=0; accum=cfg["accum_steps"]
    optimizer.zero_grad(set_to_none=True)
    for step,(x,y,w) in enumerate(loader):
        x,y,w = x.to(DEVICE,non_blocking=True), y.to(DEVICE,non_blocking=True), w.to(DEVICE,non_blocking=True)
        eff = w if apply_weights else torch.ones_like(w)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=cfg["amp"] and DEVICE.type=="cuda"):
            logits = model(x).squeeze(1)
            loss = focal_loss(logits, y, cfg["focal_gamma"], cfg["focal_alpha"], eff)/accum
        if not torch.isfinite(loss):
            skipped+=1; optimizer.zero_grad(set_to_none=True); continue
        scaler.scale(loss).backward()
        if (step+1)%accum==0 or (step+1)==len(loader):
            scaler.unscale_(optimizer)
            gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            if torch.isfinite(gn): scaler.step(optimizer)
            else: skipped+=1
            scaler.update(); optimizer.zero_grad(set_to_none=True)
        running+=loss.item()*accum; n+=1
    return running/max(n,1), skipped

@torch.no_grad()
def evaluate(model, loader, thr=0.5):
    model.eval(); P=[]; L=[]
    for x,y,_ in loader:
        x=x.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=CFG["amp"] and DEVICE.type=="cuda"):
            logit=model(x).squeeze(1)
        P.extend(torch.sigmoid(logit).float().cpu().numpy()); L.extend(y.numpy())
    P=np.array(P); L=np.array(L).astype(int); pred=(P>=thr).astype(int)
    return {"f1":f1_score(L,pred,zero_division=0),"recall":recall_score(L,pred,zero_division=0),
            "precision":precision_score(L,pred,zero_division=0),
            "roc_auc":roc_auc_score(L,P) if len(set(L))>1 else 0.0,
            "pr_auc":average_precision_score(L,P) if len(set(L))>1 else 0.0,"probs":P,"labels":L}

## 9 — Train (4-phase schedule, same LRs as the 0.96 run)

Phase 1: head only (backbone frozen), sample-weights OFF (random head → noisy weighted grads).
Phase 2: unfreeze last 2 stages, weights ON. Phase 3: full unfreeze, low backbone LR.
Phase 4: polish at min LR. NaN guard active throughout; best val-F1 checkpointed.

In [ ]:
def run_training(model, cfg):
    scaler = torch.amp.GradScaler(enabled=cfg["amp"] and DEVICE.type=="cuda")
    p1,p2,p3 = cfg["phase1_epochs"], cfg["phase2_epochs"], cfg["phase3_epochs"]
    bounds = [p1, p1+p2, p1+p2+p3]
    best_f1, no_imp = 0.0, 0
    freeze_backbone(model)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=cfg["lr_head"], weight_decay=cfg["weight_decay"])
    sched=None
    for epoch in range(cfg["total_epochs"]):
        if epoch==bounds[0]:
            unfreeze_last_n(model, n=2)
            opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr_phase2"], weight_decay=cfg["weight_decay"])
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["total_epochs"]-bounds[0], eta_min=cfg["min_lr"])
            log.info("=== Phase 2 ===")
        elif epoch==bounds[1]:
            unfreeze_last_n(model, n=None)
            opt = torch.optim.AdamW([
                {"params": model.backbone.parameters(), "lr": cfg["lr_phase3_bb"]},
                {"params": [p for n,p in model.named_parameters() if not n.startswith("backbone")],
                 "lr": cfg["lr_phase3_hd"]}], weight_decay=cfg["weight_decay"])
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["total_epochs"]-bounds[1], eta_min=cfg["min_lr"])
            log.info("=== Phase 3 (full unfreeze) ===")
        elif epoch==bounds[2]:
            for g in opt.param_groups: g["lr"]=cfg["lr_phase4"]
            log.info("=== Phase 4 (polish) ===")
        apply_w = epoch >= bounds[0]      # weights OFF in phase 1
        loss, skipped = train_one_epoch(model, train_loader, opt, scaler, cfg, apply_weights=apply_w)
        if sched: sched.step()
        m = evaluate(model, val_loader, cfg["default_threshold"])
        log.info("Ep %2d loss %.4f | val_f1 %.4f rec %.4f prec %.4f%s",
                 epoch+1, loss, m["f1"], m["recall"], m["precision"],
                 f" | skipped {skipped}" if skipped else "")
        if m["f1"] > best_f1 + cfg["early_stop_min_delta"]:
            best_f1, no_imp = m["f1"], 0
            torch.save({"state_dict":model.state_dict(),"epoch":epoch+1,"val_f1":best_f1}, cfg["checkpoint_path"])
            log.info("  ↑ saved best val_f1=%.4f", best_f1)
        else:
            no_imp += 1
            if epoch>=bounds[0] and no_imp>=cfg["early_stop_patience"]:
                log.info("Early stop @ epoch %d", epoch+1); break
    log.info("Training done. best val_f1=%.4f", best_f1)

run_training(model, CFG)

18:47:50 | Backbone frozen (head only).
18:51:01 | Ep  1 loss 0.0289 | val_f1 0.8815 rec 0.8085 prec 0.9690 | skipped 1
18:51:01 |   ↑ saved best val_f1=0.8815
18:54:16 | Ep  2 loss 0.0159 | val_f1 0.9043 rec 0.8477 prec 0.9689
18:54:17 |   ↑ saved best val_f1=0.9043
18:57:20 | Ep  3 loss 0.0142 | val_f1 0.9134 rec 0.8645 prec 0.9681
18:57:21 |   ↑ saved best val_f1=0.9134
19:00:26 | Ep  4 loss 0.0128 | val_f1 0.9150 rec 0.8596 prec 0.9779
19:00:26 |   ↑ saved best val_f1=0.9150
19:03:32 | Ep  5 loss 0.0118 | val_f1 0.9232 rec 0.8726 prec 0.9800
19:03:32 |   ↑ saved best val_f1=0.9232
19:06:42 | Ep  6 loss 0.0115 | val_f1 0.9071 rec 0.8380 prec 0.9886
19:09:53 | Ep  7 loss 0.0111 | val_f1 0.9315 rec 0.8965 prec 0.9692
19:09:53 |   ↑ saved best val_f1=0.9315
19:13:00 | Ep  8 loss 0.0110 | val_f1 0.9325 rec 0.8865 prec 0.9835
19:13:00 |   ↑ saved best val_f1=0.9325
19:16:11 | Ep  9 loss 0.0110 | val_f1 0.9314 rec 0.8852 prec 0.9828
19:19:18 | Ep 10 loss 0.0104 | val_f1 0.9270 rec 0.8784 

In [ ]:
import psutil
import os
import signal

def kill_python_processes():
    my_pid = os.getpid()  # PID of this script
    killed = []

    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        try:
            # Skip this script itself
            if proc.pid == my_pid:
                continue

            # Match processes whose name or command line contains 'python'
            if 'python' in (proc.info['name'] or '').lower() or \
               any('python' in (arg or '').lower() for arg in proc.info['cmdline'] or []):
                print(f"Killing PID {proc.pid}: {proc.info['cmdline']}")
                proc.kill()
                killed.append(proc.pid)

        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    print(f"Total killed: {len(killed)} process(es)")

if __name__ == "__main__":
    kill_python_processes()


In [21]:
ckpt = torch.load(CFG['checkpoint_path'], map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded epoch {ckpt["epoch"]} (val_f1={ckpt["val_f1"]:.4f})')
model = model.to(DEVICE)

Loaded epoch 38 (val_f1=0.9645)


In [22]:
# ═══════════════════════════════════════════════════════════════════════════
# THRESHOLD CALIBRATION
# ═══════════════════════════════════════════════════════════════════════════

def compute_metrics_at_threshold(
    labels: np.ndarray,
    probs:  np.ndarray,
    threshold: float,
) -> Dict[str, float]:
    """
    Compute all binary classification metrics given a specific threshold.

    This is the single source of truth for metric computation.
    All evaluation functions call this rather than implementing metrics inline.
    """
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()

    prec_faulty = tp / max(tp + fp, 1)
    rec_faulty  = tp / max(tp + fn, 1)
    f1_faulty   = 2 * prec_faulty * rec_faulty / max(prec_faulty + rec_faulty, 1e-9)

    try:
        roc = roc_auc_score(labels, probs)
        pr  = average_precision_score(labels, probs)
    except Exception:
        roc = pr = 0.0

    return {
        'threshold':         threshold,
        'f1_faulty':         f1_faulty,
        'f1_macro':          f1_score(labels, preds, average='macro', zero_division=0),
        'precision_faulty':  prec_faulty,
        'recall_faulty':     rec_faulty,
        'accuracy':          accuracy_score(labels, preds),
        'roc_auc':           roc,
        'pr_auc':            pr,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }

# compute_metrics_at_threshold is UNCHANGED — keep as-is.


def calibrate_threshold(
    val_probs,
    val_labels,
    min_recall=0.99,
):
    """
    Find the optimal decision threshold on the validation set.

    Two-pass search (new vs single PR-curve pass before)
    ─────────────────────────────────────────────────────
    Pass 1 — PR curve sweep:
        sklearn's precision_recall_curve gives one threshold per unique
        prediction score. Fast, but misses the optimum when score density
        is uneven (previous run found τ=0.117 when τ=0.130 gave +0.013 F1).

    Pass 2 — Fine-grained sweep at 0.002 step [0.05, 0.70]:
        Catches whatever Pass 1 missed. Runs ~325 threshold evaluations —
        negligible cost vs training time.

    Safety gate: any τ with recall_faulty < min_recall is skipped.
    If no τ passes the gate, the unconstrained best is returned with a warning.
    """
    best_f1        = 0.0
    best_tau       = 0.5
    fallback_f1    = 0.0
    fallback_tau   = 0.5

    # ── Pass 1: PR curve sweep ─────────────────────────────────────────────
    precision_arr, recall_arr, thresholds = precision_recall_curve(
        val_labels, val_probs
    )
    thresholds = np.append(thresholds, 1.0)

    for tau, _prec, rec in zip(thresholds, precision_arr, recall_arr):
        preds  = (val_probs >= tau).astype(int)
        f1_val = f1_score(val_labels, preds, zero_division=0)
        if f1_val > fallback_f1:
            fallback_f1  = f1_val
            fallback_tau = float(tau)
        if rec < min_recall:
            continue
        if f1_val > best_f1:
            best_f1  = f1_val
            best_tau = float(tau)

    # ── Pass 2: fine-grained sweep at 0.002 step ───────────────────────────
    # Catches thresholds the PR curve might space too coarsely.
    for tau in np.arange(0.05, 0.70, 0.002):
        preds  = (val_probs >= tau).astype(int)
        rec    = recall_score(val_labels, preds, zero_division=0)
        f1_val = f1_score(val_labels, preds, zero_division=0)
        if f1_val > fallback_f1:
            fallback_f1  = f1_val
            fallback_tau = float(tau)
        if rec < min_recall:
            continue
        if f1_val > best_f1:
            best_f1  = f1_val
            best_tau = float(tau)

    if best_f1 == 0.0:
        log.warning(
            'No threshold satisfies recall >= %.2f. Using unconstrained best (τ=%.4f).',
            min_recall, fallback_tau,
        )
        best_tau = fallback_tau

    best_metrics = compute_metrics_at_threshold(val_labels, val_probs, best_tau)
    log.info(
        'Calibrated threshold=%.4f | F1(FAULTY)=%.4f | '
        'Recall=%.4f | Prec=%.4f',
        best_tau, best_metrics['f1_faulty'],
        best_metrics['recall_faulty'], best_metrics['precision_faulty'],
    )
    return best_tau, best_metrics


print('Threshold calibration ready.')

Threshold calibration ready.


In [23]:
print('\n═══ Step 11: Calibrating threshold ===')
val_metrics = evaluate(model, val_loader, thr=0.5)
threshold, val_calib = calibrate_threshold(
    val_metrics['probs'], val_metrics['labels'],
    min_recall=CFG['min_recall_faulty'],
)


═══ Step 11: Calibrating threshold ===


18:26:27 | Calibrated threshold=0.2581 | F1(FAULTY)=0.9441 | Recall=0.9906 | Prec=0.9017


In [24]:
# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION & PLOTS
# ═══════════════════════════════════════════════════════════════════════════

def print_evaluation_report(metrics: dict, split: str = 'Test') -> None:
    """Human-readable evaluation summary printed to stdout."""
    sep = '─' * 62
    target_f1 = 0.98
    status = '✓ PASSED' if metrics['f1_faulty'] >= target_f1 else '✗ BELOW TARGET'
    print(f'\n{sep}')
    print(f'  BOTTLE INSPECTION — {split.upper()} SET EVALUATION')
    print(f'{sep}')
    print(f'  Decision threshold   : {metrics["threshold"]:.4f}')
    print(f'{sep}')
    print(f'  F1 (FAULTY class)    : {metrics["f1_faulty"]:.4f}   ← primary KPI')
    print(f'  Recall    (FAULTY)   : {metrics["recall_faulty"]:.4f}   ← safety metric')
    print(f'  Precision (FAULTY)   : {metrics["precision_faulty"]:.4f}')
    print(f'  F1 (macro)           : {metrics["f1_macro"]:.4f}')
    print(f'  Accuracy             : {metrics["accuracy"]:.4f}')
    print(f'  ROC-AUC              : {metrics["roc_auc"]:.4f}')
    print(f'  PR-AUC               : {metrics["pr_auc"]:.4f}')
    print(f'{sep}')
    print(f'  Confusion matrix:')
    print(f'    TP={metrics["tp"]:6d}   FP={metrics["fp"]:6d}')
    print(f'    FN={metrics["fn"]:6d}   TN={metrics["tn"]:6d}')
    print(f'{sep}')
    print(f'  F1 ≥ {target_f1:.0%} target        : {status}')
    print(f'{sep}\n')


def plot_training_history(history: dict, output_dir: str) -> None:
    """Plot train/val loss and val F1/recall over epochs."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(history['train_loss'], label='Train loss', color='steelblue')
    axes[0].plot(history['val_loss'],   label='Val loss',   color='firebrick')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history['val_f1'], color='darkorange')
    axes[1].axhline(0.98, color='green', linestyle='--', label='Target F1=0.98')
    axes[1].set_title('Val F1 (FAULTY)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(history['lr_backbone'], label='Backbone LR', color='purple')
    axes[2].plot(history['lr_head'],     label='Head LR',     color='teal')
    axes[2].legend()

    axes[2].set_title('Learning rate'); axes[2].set_yscale('log'); axes[2].grid(True, alpha=0.3)

    # Phase bands — shade each phase a different colour
    phase_colours = {1:'#e8f4f8', 2:'#fef9e7', 3:'#eafaf1', 4:'#fdf2f8'}
    for ax in axes[:3]:
        prev = 0
        for phase, colour in phase_colours.items():
            end = [CFG['phase1_epochs'],
                   CFG['phase1_epochs'] + CFG['phase2_epochs'],
                   CFG['phase1_epochs'] + CFG['phase2_epochs'] + CFG['phase3_epochs'],
                   CFG['total_epochs']][phase - 1]
            ax.axvspan(prev, min(end, len(history['val_f1'])),
                       alpha=0.25, color=colour, label=f'Ph{phase}')
            prev = end

    plt.tight_layout()
    path = os.path.join(output_dir, 'training_history.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Training history saved → {path}')


def plot_evaluation_charts(
    probs:  np.ndarray,
    labels: np.ndarray,
    threshold: float,
    output_dir: str,
) -> None:
    """Four evaluation charts saved as a single figure."""
    preds = (probs >= threshold).astype(int)
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # ── 1. Confusion matrix ───────────────────────────────────────────────
    cm = confusion_matrix(labels, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
        xticklabels=['GOOD', 'FAULTY'], yticklabels=['GOOD', 'FAULTY'],
    )
    axes[0, 0].set_title('Confusion Matrix')
    axes[0, 0].set_xlabel('Predicted'); axes[0, 0].set_ylabel('True')

    # ── 2. Precision-Recall curve ─────────────────────────────────────────
    prec_arr, rec_arr, thr_arr = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)
    axes[0, 1].plot(rec_arr, prec_arr, lw=2, color='darkorange', label=f'AP={ap:.4f}')
    axes[0, 1].axvline(
        x=recall_score(labels, preds, zero_division=0),
        color='red', linestyle='--', lw=1.2, label=f'τ={threshold:.3f}'
    )
    axes[0, 1].set_xlabel('Recall'); axes[0, 1].set_ylabel('Precision')
    axes[0, 1].set_title('Precision-Recall Curve')
    axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

    # ── 3. ROC curve ─────────────────────────────────────────────────────
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    axes[1, 0].plot(fpr, tpr, lw=2, color='steelblue', label=f'AUC={auc:.4f}')
    axes[1, 0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[1, 0].set_xlabel('FPR'); axes[1, 0].set_ylabel('TPR')
    axes[1, 0].set_title('ROC Curve')
    axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

    # ── 4. Confidence histogram ───────────────────────────────────────────
    bins = np.linspace(0, 1, 50)
    axes[1, 1].hist(probs[labels == 0], bins=bins, alpha=0.6, color='steelblue',  label='GOOD')
    axes[1, 1].hist(probs[labels == 1], bins=bins, alpha=0.6, color='firebrick',  label='FAULTY')
    axes[1, 1].axvline(threshold, color='black', linestyle='--', lw=1.5, label=f'τ={threshold:.3f}')
    axes[1, 1].set_xlabel('P(FAULTY)'); axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Confidence Score Distribution')
    axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(output_dir, 'evaluation_charts.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Evaluation charts saved → {path}')


def export_predictions(
    test_loader: DataLoader,
    probs:  np.ndarray,
    labels: np.ndarray,
    threshold: float,
    output_dir: str,
) -> pd.DataFrame:
    """
    Export a CSV with one row per test image containing:
    - image path
    - P(FAULTY) probability
    - predicted class (0/1)
    - true class (0/1)
    - human-readable predicted/true labels
    - whether the prediction was correct
    """
    preds = (probs >= threshold).astype(int)
    df    = test_loader.dataset.df.copy()

    df['prob_faulty']    = probs
    df['pred_binary']    = preds
    df['true_binary']    = labels
    df['pred_class']     = ['FAULTY' if p == 1 else 'GOOD' for p in preds]
    df['true_class']     = ['FAULTY' if l == 1 else 'GOOD' for l in labels]
    df['correct']        = (preds == labels)
    df['threshold_used'] = threshold

    path = os.path.join(output_dir, 'predictions.csv')
    df.to_csv(path, index=False)
    print(f'Predictions exported → {path}  ({len(df)} rows)')
    return df


print('Evaluation and visualisation functions ready.')

Evaluation and visualisation functions ready.


In [25]:

print('\n═══ Step 12: Final evaluation on test set ===')
test_eval    = evaluate(model, test_loader, thr=threshold)
test_metrics = compute_metrics_at_threshold(
    test_eval['labels'], test_eval['probs'], threshold
)
print_evaluation_report(test_metrics, split='Test')
preds = (test_eval['probs'] >= threshold).astype(int)
print(classification_report(
    test_eval['labels'], preds,
    target_names=['GOOD', 'FAULTY'], digits=4,
))


═══ Step 12: Final evaluation on test set ===

──────────────────────────────────────────────────────────────
  BOTTLE INSPECTION — TEST SET EVALUATION
──────────────────────────────────────────────────────────────
  Decision threshold   : 0.2581
──────────────────────────────────────────────────────────────
  F1 (FAULTY class)    : 0.9442   ← primary KPI
  Recall    (FAULTY)   : 0.9913   ← safety metric
  Precision (FAULTY)   : 0.9015
  F1 (macro)           : 0.9281
  Accuracy             : 0.9317
  ROC-AUC              : 0.9928
  PR-AUC               : 0.9952
──────────────────────────────────────────────────────────────
  Confusion matrix:
    TP=  3065   FP=   335
    FN=    27   TN=  1875
──────────────────────────────────────────────────────────────
  F1 ≥ 98% target        : ✗ BELOW TARGET
──────────────────────────────────────────────────────────────

              precision    recall  f1-score   support

        GOOD     0.9858    0.8484    0.9120      2210
      FAULTY     0

In [26]:
print('\n═══ Step 13: Saving outputs ===')
plot_evaluation_charts(
    test_eval['probs'], test_eval['labels'],
    threshold, CFG['output_dir'],
)
pred_df = export_predictions(
    test_loader, test_eval['probs'], test_eval['labels'],
    threshold, CFG['output_dir'],
)
metrics_path = os.path.join(CFG['output_dir'], 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(
        {k: round(float(v), 6) for k, v in test_metrics.items()},
        f, indent=2,
    )


═══ Step 13: Saving outputs ===
Evaluation charts saved → ./1st-krones-vision-ai-challenge/outputs_clf512_v2/evaluation_charts.png
Predictions exported → ./1st-krones-vision-ai-challenge/outputs_clf512_v2/predictions.csv  (5302 rows)


## 10 — Recall-floor table on held-out test (compare to detector 0.948 @ rec≥0.99)

In [27]:
ckpt = torch.load(CFG["checkpoint_path"], map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"]); log.info("Loaded epoch %d val_f1=%.4f", ckpt["epoch"], ckpt["val_f1"])
te = evaluate(model, test_loader, CFG["default_threshold"]); P,L = te["probs"], te["labels"]
print(f"ROC-AUC={te['roc_auc']:.4f}  PR-AUC={te['pr_auc']:.4f}\n")
print(f"{'min_rec':>8} {'best_F1':>8} {'thr':>6} {'prec':>7} {'rec':>7} {'FP':>6} {'FN':>6}")
for floor in tqdm.tqdm([0.99,0.98,0.97,0.96,0.95,0.90]):
    best=None
    for thr in np.arange(0.01,0.99,0.005):
        pred=(P>=thr).astype(int); rec=recall_score(L,pred,zero_division=0)
        if rec<floor: continue
        f1=f1_score(L,pred,zero_division=0)
        if best is None or f1>best[0]:
            best=(f1,thr,precision_score(L,pred,zero_division=0),rec,
                  int(((pred==1)&(L==0)).sum()), int(((pred==0)&(L==1)).sum()))
    print(f"{floor:>8.2f}   (no threshold reaches this recall)" if best is None else
          f"{floor:>8.2f} {best[0]:>8.4f} {best[1]:>6.3f} {best[2]:>7.4f} {best[3]:>7.4f} {best[4]:>6} {best[5]:>6}")
bf=bt=0
for thr in tqdm.tqdm(np.arange(0.01,0.99,0.005)):
    f1=f1_score(L,(P>=thr).astype(int),zero_division=0)
    if f1>bf: bf,bt=f1,thr
print(f"\nUnconstrained ceiling: F1={bf:.4f} @ thr={bt:.3f}  (the ~0.96 number is this row)")
print("Detector reference: 0.948 @ recall>=0.99, 0.954 @ best-F1.")

18:35:08 | Loaded epoch 38 val_f1=0.9645


ROC-AUC=0.9928  PR-AUC=0.9952

 min_rec  best_F1    thr    prec     rec     FP     FN


 17%|█▋        | 1/6 [00:00<00:01,  2.69it/s]

    0.99   0.9477  0.275  0.9086  0.9903    308     30


 33%|███▎      | 2/6 [00:00<00:01,  2.56it/s]

...

    0.95   0.9621  0.445  0.9637  0.9605    112    122


100%|██████████| 6/6 [00:02<00:00,  2.32it/s]


    0.90   0.9621  0.445  0.9637  0.9605    112    122


100%|██████████| 196/196 [00:00<00:00, 801.06it/s]


Unconstrained ceiling: F1=0.9621 @ thr=0.445  (the ~0.96 number is this row)
Detector reference: 0.948 @ recall>=0.99, 0.954 @ best-F1.


In [26]:
import os, json, glob, torch, numpy as np, pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── Decision threshold — from YOUR recall-floor table ──────────────────────
TAU = 0.445         # 0.290 → recall≥0.99 (F1=0.9471); 0.385 → balanced (0.9603)
OUTPUT_FILE = "submission.csv"
BATCH = 32

# Reuse the SAME preprocessing as training (ROI crop + CLAHE + eval transform).
# roi_crop_clahe, build_eval_transforms, build_model, CFG, DEVICE must already be defined.

# 1) Load the trained weights (best checkpoint)
ckpt = torch.load(CFG["checkpoint_path"], map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"] if "state_dict" in ckpt else ckpt["model"])
model.eval()
print(f"Loaded epoch {ckpt.get('epoch','?')} | τ={TAU}")

# 2) Build the test ROI map (test images have only the ROI annotation)
test_roi_map = {}
test_coco = CFG.get("test_coco_json", "./1st-krones-vision-ai-challenge/test_annotations_roi_only.json")
if Path(test_coco).exists():
    test_roi_map = load_coco_roi_map(test_coco, CFG["roi_category_id"])
print(f"Test ROI entries: {len(test_roi_map)}")

# 3) Authoritative, deterministically-ordered test file list
test_dir = Path(CFG.get("test_image_dir", "./1st-krones-vision-ai-challenge/test_images"))
submission_files = []
if Path(test_coco).exists():
    with open(test_coco) as f: tc = json.load(f)
    submission_files = [Path(im["file_name"]).name for im in tc.get("images", [])]
if not submission_files:
    submission_files = [Path(p).name for p in glob.glob(str(test_dir / "*.png"))]
def seq_key(fn):
    tail = Path(fn).stem.split("_")[-1]
    # Return tuple to handle mixed numeric/string types
    if tail.isdigit():
        return (0, int(tail))  # numeric files sort first
    else:
        return (1, fn)  # non-numeric files sort after, alphabetically
submission_files = sorted(submission_files, key=seq_key)
print(f"Test images: {len(submission_files)}")

# 4) Inference dataset — resolves extension mismatches, flags truly-missing files
class TestDS(Dataset):
    def __init__(self, files, cfg, roi_map, tdir):
        self.cfg, self.roi_map, self.tdir = cfg, roi_map, tdir
        self.items, self.missing = [], []
        for f in files:
            p = tdir / f
            if not p.exists():
                hits = list(tdir.glob(f"{Path(f).stem}.*"))
                p = hits[0] if hits else None
            (self.items if p else self.missing).append((f, p) if p else f)
        self.tf = build_eval_transforms(cfg)
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fname, path = self.items[i]
        img = roi_crop_clahe(path, self.cfg, self.roi_map)
        return self.tf(image=img)["image"], fname

ds = TestDS(submission_files, CFG, test_roi_map, test_dir)
if ds.missing:
    print(f"WARNING: {len(ds.missing)} files not found (default to 0). First: {ds.missing[:3]}")
loader = DataLoader(ds, batch_size=BATCH, shuffle=False,
                    num_workers=CFG["num_workers"], pin_memory=True)

# 5) Predict
preds = {}
with torch.no_grad():
    for x, fnames in tqdm.tqdm(loader):
        x = x.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=CFG["amp"] and DEVICE.type=="cuda"):
            p = torch.sigmoid(model(x).squeeze(1)).float().cpu().numpy()
        for fn, pf in zip(fnames, p):
            preds[fn] = int(pf >= TAU)

# 6) Write CSV in the official order; missing files default to GOOD (0)
rows = [{"image_id": f, "target": preds.get(f, 0)} for f in submission_files]
df = pd.DataFrame(rows, columns=["image_id", "target"])
df.to_csv(OUTPUT_FILE, index=False)
nf = int(df["target"].sum())
print(f"Saved {OUTPUT_FILE}: {len(df)} rows | FAULTY={nf} ({100*nf/max(len(df),1):.1f}%) | GOOD={len(df)-nf}")
print(df.head().to_string(index=False))

03:55:31 | ROI map: 4418 images | skipped 0


Loaded epoch 40 | τ=0.445
Test ROI entries: 4418
Test images: 4418


100%|██████████| 139/139 [00:39<00:00,  3.48it/s]

Saved submission.csv: 4418 rows | FAULTY=2535 (57.4%) | GOOD=1883
                                             image_id  target
64e50373-1f39-40e7-b821-655246c7a30e_000000000001.png       1
43b0c8b4-75b6-40b2-b867-ea2779f6ecec_000000000002.png       1
50269c78-2962-42a0-ac21-e83155eaeeb3_000000000003.png       0
a13c8c3d-ace7-4b58-8a25-697e6d3d4bc5_000000000004.png       0
0f06081b-03f9-4179-ab26-cd402f40e43c_000000000005.png       1


In [22]:
# os.remove('/kaggle/working/submission.csv')

In [30]:
import time, glob, statistics, torch
from pathlib import Path
from torch.utils.data import DataLoader

# Measures the classifier's per-image latency the way it actually runs: ROI crop +
# CLAHE + transform + forward. batch=1 single-image, plus a batched throughput sweep.
# Proper warmup (excludes cuDNN autotuning) and CUDA sync (else you time launches, not work).

N_WARMUP, N_MEASURE = 10, 150
test_dir = Path(CFG.get("test_dir", "/kaggle/input/competitions/1st-krones-vision-ai-challenge/test_images"))
paths = sorted(glob.glob(str(test_dir / "*.png")))[:N_WARMUP + N_MEASURE]
assert len(paths) >= N_WARMUP + N_MEASURE, f"need {N_WARMUP+N_MEASURE} images"

model.eval()
tf = build_eval_transforms(CFG)

def load_one(p):
    img = roi_crop_clahe(p, CFG, test_roi_map if 'test_roi_map' in dir() else {})
    return tf(image=img)["image"].unsqueeze(0)

# ── Single-image latency (preprocess + inference), end to end ──────────────
for p in paths[:N_WARMUP]:
    x = load_one(p).to(DEVICE)
    with torch.no_grad(), torch.amp.autocast(device_type=DEVICE.type, enabled=CFG["amp"] and DEVICE.type=="cuda"):
        model(x)
if torch.cuda.is_available(): torch.cuda.synchronize()

e2e, infer_only = [], []
for p in paths[N_WARMUP:]:
    t0 = time.perf_counter()
    x = load_one(p).to(DEVICE)                      # includes ROI crop + CLAHE + transform
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t1 = time.perf_counter()
    with torch.no_grad(), torch.amp.autocast(device_type=DEVICE.type, enabled=CFG["amp"] and DEVICE.type=="cuda"):
        model(x)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t2 = time.perf_counter()
    e2e.append((t2 - t0) * 1000)
    infer_only.append((t2 - t1) * 1000)

e2e.sort(); infer_only.sort()
med_e2e = statistics.median(e2e); med_inf = statistics.median(infer_only)
print(f"End-to-end  : median={med_e2e:6.2f}ms  p90={e2e[int(0.9*len(e2e))]:6.2f}ms  | {1000/med_e2e:5.1f} img/s")
print(f"Inference   : median={med_inf:6.2f}ms  (model forward only)")
print(f"Preprocess  : median={med_e2e-med_inf:6.2f}ms  (ROI crop + CLAHE + transform)")

# ── Batched throughput (the organizers control batch; this shows the curve) ──
class SpeedDS(torch.utils.data.Dataset):
    def __init__(self, ps): self.ps = ps
    def __len__(self): return len(self.ps)
    def __getitem__(self, i):
        img = roi_crop_clahe(self.ps[i], CFG, test_roi_map if 'test_roi_map' in dir() else {})
        return tf(image=img)["image"]

print(f"\n{'batch':>6} {'ms/img':>9} {'img/s':>9}")
for bs in [1, 8, 16, 32]:
    dl = DataLoader(SpeedDS(paths[N_WARMUP:]), batch_size=bs, num_workers=CFG["num_workers"], pin_memory=True)
    # warm
    for b in dl:
        b = b.to(DEVICE)
        with torch.no_grad(), torch.amp.autocast(device_type=DEVICE.type, enabled=CFG["amp"] and DEVICE.type=="cuda"):
            model(b)
        break
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter(); n = 0
    for b in dl:
        b = b.to(DEVICE)
        with torch.no_grad(), torch.amp.autocast(device_type=DEVICE.type, enabled=CFG["amp"] and DEVICE.type=="cuda"):
            model(b)
        n += b.shape[0]
    if torch.cuda.is_available(): torch.cuda.synchronize()
    per_img = (time.perf_counter() - t0) / n * 1000
    print(f"{bs:>6} {per_img:9.2f} {1000/per_img:9.1f}")

REQ = 70000/3600
print(f"\nLine speed: {REQ:.1f} img/s required (70k bottles/hr)")
print(f"Headroom @ single-image: {(1000/med_e2e)/REQ:.1f}x")

End-to-end  : median= 34.74ms  p90=173.68ms  |  28.8 img/s
Inference   : median= 14.34ms  (model forward only)
Preprocess  : median= 20.40ms  (ROI crop + CLAHE + transform)

 batch    ms/img     img/s
     1     19.87      50.3
     8     46.73      21.4
    16     15.21      65.7
    32     16.86      59.3

Line speed: 19.4 img/s required (70k bottles/hr)
Headroom @ single-image: 1.5x
